In [1]:
# import vars 
# import integration.upstream_cooling_exp as lib
import integration.upstream_cooling_post_lightsheet_exp as lib
import os
import numpy as np

from kexp.config import ExptParams

p = ExptParams()

In [2]:
#create dictionary of each variable to include in the parameter space, its range of values and an initial value
var_dict = {
    'detune_d2_c_mot': {
        'range': [-6.0, 0.0], 
        'initial': p.detune_d2_c_mot},
    'detune_d2_r_mot': {
        'range': [-7.0, -1.], 
        'initial': p.detune_d2_r_mot},
    'i_mot': {
        'range': [12.0, 30.0], 
        'initial': p.i_mot},
    'v_zshim_current': {
        'range': [0.0, 1.], 
        'initial': p.v_zshim_current},
    'v_xshim_current': {
        'range': [0., 5.0], 
        'initial': p.v_xshim_current},
    'v_yshim_current': {
        'range': [0.0, 5.], 
        'initial': p.v_yshim_current},
    'detune_d1_c_d1cmot': {
        'range': [2.0, 13.], 
        'initial': p.detune_d1_c_d1cmot},
    'detune_d2_r_d1cmot': {
        'range': [-6., 0.], 
        'initial': p.detune_d2_r_d1cmot},
    'pfrac_d1_c_d1cmot': {
        'range': [0.1, .99], 
        'initial': p.pfrac_d1_c_d1cmot},
    'amp_d2_r_d1cmot': {
        'range': [.02, .18], 
        'initial': p.amp_d2_r_d1cmot},
    'v_zshim_current_gm': {
        'range': [.1, 1.], 
        'initial': p.v_zshim_current_gm},
    'v_xshim_current_gm': {
        'range': [.0, 5.], 
        'initial': p.v_xshim_current_gm},
    'v_yshim_current_gm': {
        'range': [0.0, 5.], 
        'initial': p.v_yshim_current_gm},
    'pfrac_d1_c_gm': {
        'range': [0.2, .99], 
        'initial': p.pfrac_d1_c_gm},
    'pfrac_d1_r_gm': {
        'range': [0.2, 0.99], 
        'initial': p.pfrac_d1_r_gm},
    'detune_d1_c_gm': {
        'range': [3., 13.], 
        'initial': p.detune_d1_c_gm},
    'detune_d1_r_gm': {
        'range': [3., 13.], 
        'initial': p.detune_d1_r_gm},
    'pfrac_c_gmramp_end': {
        'range': [.01, .99], 
        'initial': p.pfrac_c_gmramp_end},
    'pfrac_r_gmramp_end': {
        'range': [0.01, 0.99], 
        'initial': p.pfrac_r_gmramp_end},
    'i_magtrap_init': {
        'range': [30., 160.0], 
        'initial': p.i_magtrap_init},
    'v_xshim_current_magtrap': {
        'range': [0.0, 9.], 
        'initial': p.v_xshim_current_magtrap},
    'v_yshim_current_magtrap': {
        'range': [0.0, 9.], 
        'initial': p.v_yshim_current_magtrap},
    't_magtrap': {
        'range': [0.1, 2.0], 
        'initial': p.t_magtrap},
    'v_pd_lightsheet_rampup_end': {
        'range': [3., 9.9], 
        'initial': p.v_pd_lightsheet_rampup_end},
    'i_hf_lightsheet_evap1_current': {
        'range': [192., 195.], 
        'initial': p.i_hf_lightsheet_evap1_current},
    't_hf_lightsheet_rampdown': {
        'range': [500.e-3, 2000.e-3], 
        'initial': p.t_hf_lightsheet_rampdown}
}

In [3]:
# generate lists of parameter ranges and initial values to feed to MLOOP controller

init_params = [entry['initial'] for entry in var_dict.values()]
print(init_params)

ranges = [entry['range'] for entry in var_dict.values()]
print(ranges)

var_names = list(var_dict.keys())

[-2.35, -5.5, 18.0, 0.45, 1.8, 0.86, 7.0, -5.3, 0.99, 0.05, 0.8, 0.429, 2.86, 0.8, 0.99, 7.5, 7.5, 0.05, 0.5, 40.0, 1.78, 2.0, 1.2, 7.6, 193.7, 0.5]
[[-6.0, 0.0], [-7.0, -1.0], [12.0, 30.0], [0.0, 1.0], [0.0, 5.0], [0.0, 5.0], [2.0, 13.0], [-6.0, 0.0], [0.1, 0.99], [0.02, 0.18], [0.1, 1.0], [0.0, 5.0], [0.0, 5.0], [0.2, 0.99], [0.2, 0.99], [3.0, 13.0], [3.0, 13.0], [0.01, 0.99], [0.01, 0.99], [30.0, 160.0], [0.0, 9.0], [0.0, 9.0], [0.1, 2.0], [3.0, 9.9], [192.0, 195.0], [0.5, 2.0]]


In [4]:
def main():
    #M-LOOP can be run with three commands
    
    #First create your interface
    interface = lib.CustomInterface(var_names)
    #Next create the controller. 
    #The controller must take the variables to be changed in the vars object
    #vars = Param()


    controller = lib.mlc.create_controller(interface, 
                                       controller_type = 'neural_net',  #type of controller to use, can be 'neural_network', 'gaussian process', 'differential evolution', 'random', 'nelder_mead''
                                       max_num_runs = 9500,
                                       target_cost = -109090000000,
                                       num_params = len(var_names),
                                       first_params = init_params, 
                                       min_boundary = np.transpose(ranges)[0],
                                       max_boundary = np.transpose(ranges)[1],
                                       interface_file_type = 'txt',               #file types of *exp_input.mat* and *exp_output.mat*
                                       controller_archive_file_type = 'txt',      #file type of the controller archive
                                       learner_archive_file_type = 'txt')      #file type of the learner archive
    controller.optimize()

main()

INFO     M-LOOP version 3.3.5
INFO     Optimization started.
INFO     Run: 0 (training)
INFO     params [-2.350e+00 -5.500e+00  1.800e+01  4.500e-01  1.800e+00  8.600e-01
  7.000e+00 -5.300e+00  9.900e-01  5.000e-02  8.000e-01  4.290e-01
  2.860e+00  8.000e-01  9.900e-01  7.500e+00  7.500e+00  5.000e-02
  5.000e-01  4.000e+01  1.780e+00  2.000e+00  1.200e+00  7.600e+00
  1.937e+02  5.000e-01]
0  3 values of dummy. 3 total shots. 9 total images expected.
Run ID: 76352
Acknowledged camera ready signal.
Camera is ready.
 Run ID: 76352
shot 1/3 done
shot 2/3 done
shot 3/3 done
[end_wax] called, run_id=76352
[end_wax] scope_data closed
[end_wax] cleanup_scanned complete
Device states updated.
run id 76352 complete at 2026-08-25 16:53:08  (ml_expt)
 
76352
[atomdata timing] load total=0.579s | get_data_file(initial)=0.017s | h5_open=0.011s | headers=0.031s | core_arrays=0.513s | datavault=0.001s | scope_data=0.000s
No ROI saved in run 76352 (cached).
ROI specified by Run ID. Attempting to lo

In [5]:
# Map provided parameter list onto var_dict keys (order-sensitive)
values_from_mloop = np.array([-2.55549670e+00, -5.38745062e+00,  1.80038579e+01,  4.13873054e-01,
  1.75858323e+00,  1.57647637e+00,  7.00682357e+00, -5.04156588e+00,
  7.50676801e-01,  3.38836061e-02,  6.74343595e-01,  4.82935354e-01,
  2.51724831e+00,  7.29308261e-01,  9.71981943e-01,  7.58588988e+00,
  7.58062947e+00,  1.68877396e-02,  5.00711161e-01,  4.74435150e+01,
  1.66793727e+00,  1.66101841e+00,  1.13549237e+00,  6.98855680e+00,
  1.93609118e+02,  5.04042004e-01], dtype=float)

# ensure the lengths match before zipping
if len(values_from_mloop) != len(var_names):
    raise ValueError(f"Expected {len(var_names)} values, got {len(values_from_mloop)}")

# build mapping in declared var_dict order
mapped_params = dict(zip(var_names, values_from_mloop))

# print in the requested p.<name> format
for name, value in mapped_params.items():
    print(f"self.p.{name} = {value}")

self.p.detune_d2_c_mot = -2.5554967
self.p.detune_d2_r_mot = -5.38745062
self.p.i_mot = 18.0038579
self.p.v_zshim_current = 0.413873054
self.p.v_xshim_current = 1.75858323
self.p.v_yshim_current = 1.57647637
self.p.detune_d1_c_d1cmot = 7.00682357
self.p.detune_d2_r_d1cmot = -5.04156588
self.p.pfrac_d1_c_d1cmot = 0.750676801
self.p.amp_d2_r_d1cmot = 0.0338836061
self.p.v_zshim_current_gm = 0.674343595
self.p.v_xshim_current_gm = 0.482935354
self.p.v_yshim_current_gm = 2.51724831
self.p.pfrac_d1_c_gm = 0.729308261
self.p.pfrac_d1_r_gm = 0.971981943
self.p.detune_d1_c_gm = 7.58588988
self.p.detune_d1_r_gm = 7.58062947
self.p.pfrac_c_gmramp_end = 0.0168877396
self.p.pfrac_r_gmramp_end = 0.500711161
self.p.i_magtrap_init = 47.443515
self.p.v_xshim_current_magtrap = 1.66793727
self.p.v_yshim_current_magtrap = 1.66101841
self.p.t_magtrap = 1.13549237
self.p.v_pd_lightsheet_rampup_end = 6.9885568
self.p.i_hf_lightsheet_evap1_current = 193.609118
self.p.t_hf_lightsheet_rampdown = 0.504042004


In [ ]:
#### uncomment this and run it if u wanna delete the logs


# import os, shutil
# # Specify the path of the file to be deleted
# folder = r'C:\Users\bananas\code\k-exp\kexp\experiments\Mloop testing\M-LOOP_archives'
# file_path_2 = r'C:\Users\bananas\code\k-exp\kexp\experiments\Mloop testing\M-LOOP_logs'

# for filename in os.listdir(folder):
#     file_path = os.path.join(folder, filename)
#     try:
#         if os.path.isfile(file_path) or os.path.islink(file_path):
#             os.unlink(file_path)
#         elif os.path.isdir(file_path):
#             shutil.rmtree(file_path)
#     except Exception as e:
#         print('Failed to delete %s. Reason: %s' % (file_path, e))

# for filename in os.listdir(file_path_2):
#     file_path = os.path.join(folder, filename)
#     try:
#         if os.path.isfile(file_path) or os.path.islink(file_path):
#             os.unlink(file_path)
#         elif os.path.isdir(file_path):
#             shutil.rmtree(file_path)
#     except Exception as e:
#         print('Failed to delete %s. Reason: %s' % (file_path, e))